# 교안 01-5: 코드 실행 MCP 서버 붙이기

## 핵심 목표

모델이 **스스로 코드를 짜서 실행**하게 만들어, 자릿수 큰 계산이나 통계를 검증 가능한 값으로 답하게 한다.

## 학습 순서

1. 코드 실행 서버(mcp-server-code-runner) 연결과 `run-code` 도구
2. 코드를 문자열로 넘겨 실행하고 출력 받기
3. `print` 가 없으면 아무것도 돌아오지 않는 이유
4. 여러 줄 코드 보내기
5. 모델이 스스로 코드를 짜서 계산하게 하기

## 쓰는 MCP 서버와 공식 문서

| 서버 | 실행 | 전송 | 공식 문서 |
|---|---|---|---|
| 코드 실행 `mcp-server-code-runner` | `npx` | stdio | https://github.com/formulahendry/mcp-server-code-runner |

## 준비물

- **Node.js**(`npx -v`). 없으면 https://nodejs.org
- **에이전트를 만드는 절부터 `OPENAI_API_KEY`** 가 필요합니다(일차 폴더의 `.env`).

---
## 준비

`quiet_stdio_logs()` 를 먼저 부릅니다. 이 서버는 시작할 때 `Code Runner MCP Server running on stdio` 같은
**사람용 안내문**을 표준출력에 찍는데, MCP 규약상 표준출력은 JSON 전용이라 클라이언트가 그 줄을 읽으려다 실패하며
긴 트레이스백을 남깁니다. 동작에는 문제가 없고 화면만 어지럽히므로 **그 로거만** 조용히 시킵니다
(남의 서버를 우리가 고칠 수는 없습니다).

In [ ]:
from pathlib import Path

# 노트북에는 __file__ 이 없다. 주피터는 노트북이 있는 폴더를 작업 폴더로 잡아 주므로 그 위가 일차 폴더다.
DAY_DIR = Path.cwd().parent        # 일차 폴더(day21). 아래 경로들의 기준점
sys.path.append(str(DAY_DIR))   # 일차 폴더의 utils.py 를 쓴다

import os
import sys
from pprint import pprint

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

from utils import block_text, load_api_key, print_trajectory, quiet_stdio_logs

quiet_stdio_logs()      # 이 서버가 stdout 에 섞어 보내는 안내문 때문에 나는 긴 경고를 끈다
load_api_key(DAY_DIR)   # 모델을 부르는 절이 있으므로 키를 맨 앞에서 확인한다

print("준비 완료. 일차 폴더:", DAY_DIR)

---
## 서버 설정: 이번에는 `env` 가 하나 더 붙는다

앞의 두 서버와 달리 이 설정에는 **`env`** 키가 있습니다. 서버가 **다른 프로그램(파이썬)을 다시 실행**하기 때문입니다.

| 키 | 값 | 뜻 |
|---|---|---|
| `command` | `"npx"` | 서버를 띄울 실행기(Node 패키지) |
| `args[0]` | `"-y"` | 설치 여부를 묻지 않고 진행 |
| `args[1]` | `"mcp-server-code-runner"` | 띄울 서버 패키지 이름 |
| `transport` | `"stdio"` | 자식 프로세스로 띄우고 표준입출력으로 대화 |
| `env` | `CHILD_ENV` | **서버에게 물려줄 환경 변수**. 서버가 `python` 을 찾을 수 있게 `PATH` 를 맞춰 준다 |

`env` 를 왜 손봐야 하는지가 핵심입니다. 이 서버는 코드를 임시 파일에 쓰고 **`python <임시파일>`** 을 실행합니다.
그런데 맥에는 `python` 이라는 이름 없이 `python3` 만 있는 경우가 많아 그대로면 실행이 실패합니다.
그래서 **지금 이 노트북이 도는 가상환경의 `bin` 폴더를 `PATH` 맨 앞에 붙여** 넘깁니다.
결과적으로 이 한 줄이 "**코드를 어느 환경에서 실행할지**" 를 정합니다(그 환경의 pandas·numpy 를 쓰게 됩니다).

`os.environ` 을 통째로 넘기지 않는 이유는 두 가지입니다. 셸 프롬프트(`PS1`) 같은 변수까지 딸려가 경고가 뜨고,
어느 `python` 이 잡힐지 통제되지 않습니다.

In [ ]:
# utils.child_env() 가 만드는 것과 같은 값을 여기서 눈으로 보며 만든다.
CHILD_ENV = {
    # 지금 커널이 도는 가상환경의 bin 폴더를 PATH 맨 앞에 둔다 -> 이 환경의 python 이 잡힌다.
    # 뒤에 기존 PATH 를 이어 붙이는 이유: 서버를 띄우는 npx·node 도 PATH 에서 찾기 때문이다.
    "PATH": str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", ""),
    "HOME": os.environ.get("HOME", ""),
    "TMPDIR": os.environ.get("TMPDIR", ""),
}
CHILD_ENV = {key: value for key, value in CHILD_ENV.items() if value}   # 빈 값은 넘기지 않는다

CODE_RUNNER = {
    "command": "npx",                          # Node 패키지 실행기
    "args": ["-y", "mcp-server-code-runner"],  # 묻지 않고 진행 + 띄울 서버 패키지 이름
    "transport": "stdio",                      # 내 컴퓨터에 프로세스로 띄운다
    "env": CHILD_ENV,                          # 위에서 만든 환경 변수(서버가 python 을 찾게 한다)
}

print("코드를 실행할 환경:", CHILD_ENV["PATH"].split(os.pathsep)[0])

---
## 1. 코드 실행 서버에 붙기

도구 이름이 **하이픈이 든 `run-code`** 입니다. 파이썬 변수 이름으로는 쓸 수 없으므로 딕셔너리에서 꺼내 씁니다.

In [ ]:
print("서버를 띄우는 중입니다(첫 실행은 오래 걸립니다)...")
client = MultiServerMCPClient({"code": CODE_RUNNER})
tools = await client.get_tools(server_name="code")
by_name = {tool.name: tool for tool in tools}

run_code = by_name["run-code"]      # 도구 이름은 서버가 정한다. 점 표기 대신 딕셔너리로 꺼낸다.

print(f"도구 {len(tools)}개")
for tool in tools:
    print(f" - {tool.name}({', '.join(tool.args)}): {tool.description.strip().splitlines()[0][:60]}")

---
## 2. 코드를 넘겨 실행하기

`languageId` 로 어떤 언어인지 알려 주면 서버가 그 언어의 실행기를 찾아 돌립니다.
파이썬만 되는 서버가 아니라, 그래서 이 인자가 있습니다.

In [ ]:
snippet = "print(sum([1, 2, 3, 4, 5]))"

print("보낸 코드 :", snippet)
print("받은 결과 :", block_text(await run_code.ainvoke({"code": snippet, "languageId": "python"})))

---
## 3. 출력하지 않으면 아무것도 돌아오지 않는다

이 서버가 돌려주는 것은 **표준 출력뿐**입니다. 값을 계산만 하고 `print` 하지 않으면 **빈 결과**가 옵니다.
같은 계산을 두 번 보내 나란히 확인합니다.

In [ ]:
pprint(await run_code.ainvoke({"code": "1 + 1", "languageId": "python"}))          # print 없음
pprint(await run_code.ainvoke({"code": "print(1 + 1)", "languageId": "python"}))   # print 있음

> 이 성질은 뒤에서 **에이전트에게 반드시 알려 줘야 할 규칙**이 됩니다.
> 모델이 `result = ...` 로만 끝내면 값이 안 돌아와서, 모델은 "계산이 실패했다"고 착각하고 엉뚱한 답을 냅니다.

---
## 4. 여러 줄 코드도 그대로 보낸다

줄바꿈(`\n`)을 이어 붙이면 여러 줄짜리 코드가 됩니다. 파일로 저장할 필요가 없습니다.

In [ ]:
# 사람이 암산하기 어려운 값이라 '코드로 계산했는지'가 눈에 보인다.
sales = [318000, 274500, 391200, 288900, 350100, 412700, 299800]

stats_code = (
    "import statistics\n"
    f"sales = {sales}\n"                                          # 값을 코드 문자열 안에 박아 넣는다
    "print('평균:', round(statistics.mean(sales), 1))\n"
    "print('표준편차:', round(statistics.pstdev(sales), 1))"       # pstdev = 모표준편차
)

print(stats_code)
print("-" * 40)
print(block_text(await run_code.ainvoke({"code": stats_code, "languageId": "python"})))

> **호출마다 새 프로세스입니다.** 앞 호출에서 만든 변수는 다음 호출에 남지 않습니다.
> 그래서 위 코드도 `sales` 를 **매번 코드 안에 다시 적어** 넘겼습니다. 이 성질은 뒤에서 다시 만납니다.

### 🖐️ 함께 따라하기: 다른 데이터로, 다른 통계를 코드로 계산하기

데모는 편의점 7일치 매출의 평균·표준편차를 구했습니다. 이번엔 **카페 하루 주문 건수**로 **다른 통계**를 구합니다.

```python
orders = [42, 51, 38, 67, 45, 73, 58, 49, 61, 55]
```

1. 위 리스트를 코드 문자열 안에 넣어 `run-code` 로 보내세요.
2. `statistics` 모듈로 **중앙값(`median`)** 과 **최댓값·최솟값의 차이**를 구해 `print` 하세요.
3. 반환값을 `block_text()` 로 출력하세요.
4. 같은 코드에서 `print` 를 하나 빼고 다시 보내, **그 값이 사라지는지** 확인하세요.

**확인 기준**: 중앙값 `53.0`, 최댓값과 최솟값의 차이 `35` 가 나옵니다.
`print` 를 뺀 쪽은 그 줄의 값이 **결과에서 사라집니다**.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) orders 리스트를 코드 문자열 안에 넣어 run-code 로 보낸다
# 2) statistics.median 과 max-min 을 print 로 출력하는 코드를 짠다
# 3) block_text 로 결과를 출력한다
# 4) print 를 하나 빼고 다시 보내 값이 사라지는지 확인한다

---
## 5. 에이전트에 붙이기: 모델이 스스로 코드를 짠다

시스템 프롬프트에서 두 가지를 못 박습니다.

- **"계산은 코드로"**: 안 그러면 모델이 암산으로 답해 버립니다.
- **"결과는 `print` 로"**: 앞에서 본 그대로입니다. 출력하지 않으면 빈 결과가 돌아옵니다.

한 줄로 끝나지 않습니다. 모델은 결과만 보고는 **왜 비었는지 알 길이 없어 같은 코드를 계속 다시
보냅니다**. 이 반복이 실습 중 화면이 멈춘 것처럼 보이는 가장 흔한 원인입니다(요청이 쌓여 분당 토큰
한도까지 먹습니다). 그래서 **결과가 비면 `print` 를 빠뜨린 것이니 `print` 를 넣어 다시 실행하라**고
프롬프트에 미리 적어 둡니다.

그래도 모델이 반복에 빠질 수 있으니 **`ModelCallLimitMiddleware`** 로 모델 호출 횟수에 상한을 둡니다.
`exit_behavior="end"` 면 상한에서 **예외 없이 스스로 끝나고** 여태 기록을 그대로 돌려줍니다.
`ChatOpenAI(timeout=60)` 은 요청 하나가 늦어질 때의 상한입니다(기본값은 10분이라 사실상 없는 것과 같습니다).

In [ ]:
# timeout 을 준다. 기본값은 요청 하나를 10분까지 기다리고 두 번 더 재시도해서,
# 응답이 늦거나 분당 한도에 걸리면 화면만 보고는 멈춘 것과 구별되지 않는다.
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=60)

agent = create_agent(
    model,
    tools,                      # 서버에서 받은 도구를 그대로 붙인다
    # 이 서버의 성질을 프롬프트에 미리 적어 둔다. 안 적어 두면 모델이 이유를 모른 채
    # 같은 코드를 계속 다시 보낸다(화면이 멈춘 것처럼 보인다).
    system_prompt=(
        "너는 데이터 분석 도우미다. 수치 계산은 반드시 코드 실행 도구로 계산한 뒤 그 결과로 답한다. "
        "코드의 결과에 대한 마지막 줄은 반드시 print 로 출력한다. "
        "결과가 비어 있으면 print 를 빠뜨린 것이니 print 를 넣어 다시 실행한다. "
    ),
    # 반복에 빠져도 상한에서 스스로 끝난다(예외 없이 기록을 돌려준다).
    middleware=[ModelCallLimitMiddleware(run_limit=8, exit_behavior="end")],
)

question = (
    f"다음 7일치 매출 {sales} 의 평균과 표준편차(모표준편차)를 구하고, "
    "평균보다 큰 날이 며칠인지 알려 줘. 암산하지 말고 반드시 코드를 실행해서 계산해."
)
print("질문:", question, "\n")

result = await agent.ainvoke({"messages": question})
print_trajectory(result)

> 기록에서 **모델이 직접 쓴 코드**를 읽어 보세요. 우리가 짠 코드가 아닙니다.
> 앞에서 우리가 손으로 만든 문자열을 이제 모델이 만들어 보냅니다. 값이 맞는지도 실행 결과로 검증됩니다.

> 도구 결과에 **"표준 출력이 비어 있습니다..."** 안내가 찍히고, 그다음 호출에서 모델이
> `print` 를 붙여 다시 보내는 모습이 자주 보입니다. 위에서 감싼 도구가 일한 자리입니다.
> 이 안내가 없다면 모델은 같은 코드를 계속 다시 보내며 멈춘 것처럼 보였을 것입니다.
> 모델이 도구를 한 번에 두 번 부르기도 해서, 안내와 호출이 짝을 지어 두 줄씩 보일 수 있습니다.

> 중간에 `NameError` 로 한 번 실패하고 다시 실행하는 모습도 보일 수 있습니다.
> 앞 호출의 `import` 가 남지 않는데 모델이 그걸 잊었을 때 나는 오류이고, 대개 코드를 온전히 다시 써서 복구합니다.

### 🖐️ 함께 따라하기: 도구 없이 시켰을 때와 비교하기

정말 코드 실행 때문에 답이 맞는 것인지 확인하려면 **도구 없는 에이전트**와 비교해야 합니다.
사람도 암산하기 어려운 큰 수로 물어봅니다.

1. `create_agent(model, [], system_prompt="너는 계산 도우미다. 답만 간결히 말하라.")` 로 도구 없는 에이전트를 만드세요.
2. `"384729 * 573921 은 얼마인가? 그리고 그 값을 7로 나눈 나머지는?"` 을 물어보세요(`ainvoke` 로).
3. 같은 질문을 **위에서 만든 `agent`**(코드 실행 도구가 붙은 쪽)에도 던지세요.
4. 두 답을 파이썬으로 직접 계산한 값과 비교해 출력하세요.

**확인 기준**: 파이썬으로 계산한 정답은 `220804052409`, 나머지는 `3` 입니다.
도구 없는 쪽은 **자신 있게 틀린 숫자**를 내놓기 쉽습니다(맞을 때도 있지만 근거가 없습니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 도구가 빈 에이전트를 만든다
# 2) 큰 수 곱셈과 나머지를 물어본다
# 3) 코드 실행 도구가 붙은 agent 에도 같은 질문을 던진다
# 4) 파이썬으로 직접 계산한 값과 비교해 출력한다

---
## 이번 실습 정리

| 배운 것 | 요점 |
|---|---|
| `env` 인자 | 서버가 다른 프로그램을 실행할 때는 `PATH` 를 맞춰 넘긴다. 이것이 곧 실행 환경 선택 |
| `run-code` | `code`·`languageId` 를 받아 **표준 출력**을 돌려준다. `print` 가 없으면 빈 결과 |
| 무상태 | 호출마다 새 프로세스. 각 호출의 코드는 그것만으로 완결되어야 한다 |
| 에이전트 | "계산은 코드로, 결과는 print 로" 를 시스템 프롬프트에 못 박는다 |
| 안전 | 남의 코드를 내 권한으로 실행하는 구조. 실무에서는 격리된 실행기를 쓴다 |

다음 실습: `06_종합_데이터분석_자동화.ipynb` 에서 세 서버를 한꺼번에 붙여,
질문 한 문장에서 리포트 파일까지 이어지는 파이프라인을 만듭니다.